# Group 6: LoRA fine-tuning of MedPsy-1.7B with Unsloth

Same data, same LoRA settings and same scoring as our first run (Hugging Face PEFT). Only the training tool is different,
so we can compare the two.

**How to run**
1. Runtime > Change runtime type > **T4 GPU**.
2. Runtime > **Run all**. Upload `ft_data_unsloth.zip` when asked.
3. Part A trains (about 20-30 min), then **restarts the Python session on purpose** (a "session crashed/restarted" message is expected).
4. Click the first cell of **Part B**, then Runtime > **Run cell and below**. Part B scores the model, makes the GGUF file
   and downloads the results to your laptop (allow multiple downloads if the browser asks).

Nothing is saved to Google Drive. If Colab disconnects in the middle, just run it again.

# Part A: train with Unsloth

In [ ]:
!pip -q install unsloth

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported   # Unsloth must be imported before transformers
import json, math, os, random, time, zipfile
import torch
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

MODEL_NAME = "qvac/MedPsy-1.7B"
SEED = 6
END = "<|im_end|>"
random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime to T4 GPU")

In [ ]:
# Upload ft_data_unsloth.zip (train, val, holdout, the explain prompts, and the replay answers from the first run).
from google.colab import files
uploaded = files.upload()
zipfile.ZipFile(list(uploaded)[0]).extractall("/content/data")


def load(name):
    with open(f"/content/data/{name}.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train = load("train")
val = load("val")
replay = load("replay")
print("train", len(train), "| val", len(val), "| replay", len(replay))

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=2048,
                                                     dtype=None, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()
print("vocabulary size:", len(tokenizer), "(should be the same as the original model)")

Same settings as the first run: 3 epochs, learning rate 2e-4, batch of 8, linear schedule with 5% warmup,
and the loss only on the reply (`{"answer": N}<|im_end|>`), not on the prompt.

In [ ]:
def encode(example):
    prompt_ids = tokenizer(example["prompt"], add_special_tokens=False)["input_ids"]
    reply_ids = tokenizer(example["completion"] + END, add_special_tokens=False)["input_ids"]
    return {"input_ids": prompt_ids + reply_ids,
            "attention_mask": [1] * (len(prompt_ids) + len(reply_ids)),
            "labels": [-100] * len(prompt_ids) + reply_ids}


train_set = Dataset.from_list([encode(r) for r in train + replay]).shuffle(seed=SEED)
val_set = Dataset.from_list([encode(r) for r in val])
total_steps = 3 * math.ceil(len(train_set) / 8)   # batch of 8 = 4 per step x 2 steps of gradient accumulation

settings = TrainingArguments(
    output_dir="/content/unsloth_run", num_train_epochs=3, learning_rate=2e-4,
    per_device_train_batch_size=4, gradient_accumulation_steps=2, per_device_eval_batch_size=4,
    lr_scheduler_type="linear", warmup_steps=int(0.05 * total_steps), weight_decay=0.0, max_grad_norm=1.0,
    fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), optim="adamw_torch",
    logging_steps=25, eval_strategy="epoch", save_strategy="no", seed=SEED, report_to="none")
trainer = Trainer(model=model, args=settings, train_dataset=train_set, eval_dataset=val_set,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))

started = time.time()
trainer.train()
minutes = round((time.time() - started) / 60, 1)

history = [{"epoch": round(log["epoch"]), "val_loss": log["eval_loss"]}
           for log in trainer.state.log_history if "eval_loss" in log]
print("training minutes:", minutes)
print(history)
model.save_pretrained("/content/adapter_unsloth")
tokenizer.save_pretrained("/content/adapter_unsloth")
json.dump({"tool": "unsloth", "training_minutes": minutes, "history": history,
           "peak_gpu_memory_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2)},
          open("/content/unsloth_history.json", "w"), indent=1)

In [ ]:
# Restart Python so Part B scores with plain Hugging Face code, exactly like the first run.
# Colab will say the session crashed or restarted: that is expected. Then run Part B.
os.kill(os.getpid(), 9)

# Part B: score and convert (same code as the first run)
Click the first code cell below, then Runtime > **Run cell and below**.

In [ ]:
import json, math, os, random, re, time, zipfile
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

MODEL_NAME = "qvac/MedPsy-1.7B"
SEED = 6
random.seed(SEED)
torch.manual_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime to T4 GPU")


def load(name):
    with open(f"/content/data/{name}.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


val = load("val")
holdout = load("holdout")
keep_prompts = load("keep_prompts")
explain_before = json.load(open("/content/data/explain_before.json"))
SAVE = "/content/results_unsloth"
os.makedirs(SAVE, exist_ok=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to("cuda")
model = PeftModel.from_pretrained(model, "/content/adapter_unsloth")
model.eval()
END = "<|im_end|>"


def ids_of(text):
    return tokenizer(text, add_special_tokens=False)["input_ids"]

In [ ]:
def allowed_replies(example):
    if '"yes"' in example["completion"] or '"no"' in example["completion"]:
        return ['{"answer": "yes"}', '{"answer": "no"}']
    user = example["prompt"].split("<|im_start|>user\n")[-1]
    option_count = len(re.findall(r"^\d+\. ", user.split("Options:\n")[1], flags=re.M)) - 1
    return ['{"answer": ' + str(n) + "}" for n in range(option_count + 1)]


def padded_batch(pairs):
    # pairs = [(prompt_ids, reply_ids), ...]. Pads on the LEFT so every reply ends at the same place.
    rows = [prompt + reply for prompt, reply in pairs]
    longest = max(len(r) for r in rows)
    input_ids = torch.full((len(rows), longest), tokenizer.pad_token_id)
    mask = torch.zeros((len(rows), longest), dtype=torch.long)
    for i, r in enumerate(rows):
        input_ids[i, longest - len(r):] = torch.tensor(r)
        mask[i, longest - len(r):] = 1
    positions = (mask.cumsum(dim=1) - 1).clamp(min=0)
    return input_ids.to(model.device), mask.to(model.device), positions.to(model.device)


def reply_log_probs(pairs):
    # Log-probability of each reply token. Only the last few positions get logits (saves a lot of memory).
    input_ids, mask, positions = padded_batch(pairs)
    keep = max(len(reply) for _, reply in pairs) + 1
    with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
        logits = model(input_ids=input_ids, attention_mask=mask, position_ids=positions,
                       logits_to_keep=keep).logits.float()
    log_probs = torch.log_softmax(logits, dim=-1)
    length = input_ids.shape[1]
    first_kept = length - keep
    results = []
    for i, (_, reply) in enumerate(pairs):
        token_scores = []
        for position in range(length - len(reply), length):
            token_scores.append(log_probs[i, position - 1 - first_kept, input_ids[i, position]])
        results.append(torch.stack(token_scores))
    return results


@torch.no_grad()
def model_reply(example):
    replies = allowed_replies(example)
    prompt_ids = ids_of(example["prompt"])
    pairs = [(prompt_ids, ids_of(reply + END)) for reply in replies]
    scores = [s.sum().item() for s in reply_log_probs(pairs)]
    return replies[scores.index(max(scores))]


def is_nothing(reply):
    return reply in ('{"answer": 0}', '{"answer": "no"}')


def score(examples, label):
    groups = {}
    wrong_rows = []
    for example in examples:
        got = model_reply(example)
        want = example["completion"]
        meta = example["meta"]
        group_name = meta["mode"] + " / " + meta["step"]
        for name in ("ALL", group_name):
            g = groups.setdefault(name, {"n": 0, "right": 0, "made_up": 0, "nothing_cases": 0, "missed": 0, "something_cases": 0})
            g["n"] += 1
            g["right"] += int(got == want)
            if is_nothing(want):
                g["nothing_cases"] += 1
                g["made_up"] += int(not is_nothing(got))
            else:
                g["something_cases"] += 1
                g["missed"] += int(is_nothing(got))
        if got != want:
            wrong_rows.append({"id": meta["id"], "group": group_name, "want": want, "got": got})
    print(f"{label}")
    print(f"  {'group':28s} {'n':>5s} {'right':>7s} {'made up':>9s} {'missed':>8s}")
    for name in sorted(groups, key=lambda k: (k != "ALL", k)):
        g = groups[name]
        made_up = 100 * g["made_up"] / g["nothing_cases"] if g["nothing_cases"] else 0
        missed = 100 * g["missed"] / g["something_cases"] if g["something_cases"] else 0
        print(f"  {name:28s} {g['n']:5d} {100 * g['right'] / g['n']:6.1f}% {made_up:8.1f}% {missed:7.1f}%")
    return {"groups": groups, "wrong": wrong_rows}

In [ ]:
after = {"val": score(val, "UNSLOTH AFTER - validation"),
         "holdout": score(holdout, "UNSLOTH AFTER - holdout (10 unseen complaints)")}
json.dump(after, open(SAVE + "/scores_after_unsloth.json", "w"), indent=1)
print("first run (PEFT) for comparison: validation 97.2%, holdout 94.1%; before training: 73.6% and 68.4%")

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=60):
    input_ids = torch.tensor([ids_of(prompt)]).to(model.device)
    with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
        output = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                                eos_token_id=tokenizer.convert_tokens_to_ids(END),
                                pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True).strip()


def first_sentence(text):
    text = " ".join(text.split("</think>")[-1].split())
    match = re.search(r"[.!?](\s|$)", text)
    return text[:match.end()].strip() if match else text


explain_after = []
for row in keep_prompts[:20]:
    explain_after.append({"text": row["meta"]["text"], "answer": first_sentence(generate(row["prompt"]))})
json.dump(explain_after, open(SAVE + "/explain_after_unsloth.json", "w"), indent=1)
same = sum(1 for b, a in zip(explain_before, explain_after) if b["answer"] == a["answer"])
print(same, "of 20 explanations identical to the original model")

In [ ]:
merged = model.merge_and_unload()
merged = merged.to(torch.float16)
merged.save_pretrained("/content/medpsy-ft", safe_serialization=True)
tokenizer.save_pretrained("/content/medpsy-ft")
for name in ("generation_config.json",):
    if os.path.exists(f"/content/medpsy-ft/{name}"):
        print("saved", name)
del merged, model
torch.cuda.empty_cache()

In [ ]:
!git clone -q --depth 1 --branch b12447 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install sentencepiece
!python /content/llama.cpp/convert_hf_to_gguf.py /content/medpsy-ft --outtype f16 --outfile /content/medpsy-ft-f16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_NATIVE=ON > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/medpsy-ft-f16.gguf /content/medpsy-1.7b-unsloth-q4_k_m.gguf Q4_K_M

In [ ]:
# Download the results to the laptop. The model file is 1.1 GB, so it takes a few minutes.
# If the browser blocks it: open the folder icon on the left, right-click the file, Download.
import shutil
shutil.copy("/content/unsloth_history.json", SAVE)
shutil.make_archive("/content/results_unsloth", "zip", SAVE)
from google.colab import files
files.download("/content/results_unsloth.zip")
files.download("/content/medpsy-1.7b-unsloth-q4_k_m.gguf")